### 해양수산부 국립해양조사원_국가해양관측망 실측 파랑
*from 해양수산부 국립해양조사원*
>국가해양관측망(해양관측부이, 해양과학기지 등)에서 관측하는 파랑(파고, 파주기, 파향) 데이터를 제공 기능

- **제공기관:** 해양수산부 국립해양조사원
- **API 유형:** REST	데이터포맷	JSON+XML
- **End Point:** *https://apis.data.go.kr/1192136/noonWave*
- **Service:**  */GetNoonWaveApiService*  [ 파랑(파고, 파주기, 파향) 데이터 ]

| 항목명 (Field) | 타입 (Type) | 설명 (Description)|
| :--- |:----------| :--- |
| `obsvtrNm` | string    | 관측소명 |
| `lot` | number    | 관측소 경도 |
| `lat` | number    | 관측소 위도 |
| `obsrvnDt` | string    | 관측일시 |
| `wvhgt` | number    | 파고(m) |
| `wvpd` | number    | 파주기(sec) |
| `wvdrct` | number    | 파향(deg) |
| `maxWvhgt` | number    | 최대파고(m) |
| `maxWvpd` | number    | 최대파주기(sec) |

In [89]:
%useLatestDescriptors
%use dataframe

In [ ]:
@file:DependsOn("org.xerial:sqlite-jdbc:3.49.1.0")
import java.sql.DriverManager
Class.forName("org.sqlite.JDBC")

In [111]:
val serviceKeyFilePath = "/Users/unchil/.EnvDataCollector/envDataCollector.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

**[관측소 출처: 오픈API 활용가이드_국가해양관측망 실측 파랑.hwp](https://www.data.go.kr/cmm/cmm/fileDownload.do?atchFileId=FILE_000000003610389&fileDetailSn=1)**

In [90]:
val filePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/KotlinNotebooks/data/wave_Observatory_list.csv"
val df_observatory = DataFrame.readCsv(filePath, ',')

In [105]:
df_observatory.columnNames()

[code, name]

In [116]:
val tableName_Observatory = "WaveInfoObservatory"
val sql_Observatory = """INSERT INTO $tableName_Observatory (code, name) VALUES (?,?)""".trimIndent()

DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql_Observatory)?.use { pstmt ->
        df_observatory.forEach { row ->
            try {
                pstmt.setString(1, row["code"].toString())
                pstmt.setString(2, row["name"].toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}

In [126]:
val endPoint = configData.WAVE_INFO.endPoint
val serviceKey = configData.WAVE_INFO.apikey
val subPath = configData.WAVE_INFO.subPath
val numOfRows = 10
val type = "json"
val min = "5"

* 5분 주기로 수집 ( 마지막 페이지의 결과만 취합)

In [ ]:
val allDataFrames = mutableListOf<DataFrame<*>>()

df_observatory.forEach { row ->
    val obsCode = row["code"].toString()
    val firstUrl = "${endPoint}/${subPath}?serviceKey=${serviceKey}&pageNo=1&numOfRows=1&type=$type&regDate=${reqDate}&min=$min&obsCode=$obsCode"
    try{
        val df_first = DataFrame.readJson(firstUrl)
        val totalCnt = df_first["body"]["totalCount"][0].toString().toInt()
        val lastPage = if ((totalCnt % numOfRows) == 0) totalCnt / numOfRows else (totalCnt / numOfRows) + 1
       // println( "obsCode:$obsCode, resultCode:${df_first["header"]["resultCode"][0]}, totalCount:${df_first["body"]["totalCount"][0]}, lastPage:${lastPage}")
        val lastUrl = "${endPoint}/${subPath}?serviceKey=${serviceKey}&pageNo=${lastPage}&numOfRows=${numOfRows}&type=$type&regDate=${reqDate}&min=$min&obsCode=$obsCode"
     //   println( lastUrl)
        val df_last = DataFrame.readJson(lastUrl)
        val data = df_last["body"]["items"]["item"].first() as DataFrame<*>
        allDataFrames.add(data)
    }catch(e:Exception){
        println("errMsg:[${e.localizedMessage}],obsCode:[$obsCode], [${firstUrl}]")

    }
}

In [149]:
val df = allDataFrames.concat()
val df_cleaned = df.remove("ctnWvhgt", "ctnWvpd").dropNulls( "wvhgt" )
df_cleaned

obsvtrNm,lot,lat,obsrvnDt,wvdrct,maxWvhgt,maxWvpd,wvhgt,wvpd
이어도,125.182220,32.122768,2026-10-07 11:40,42.700001,2.100000,null,1.300000,5.700000
신안가거초,124.592773,33.941940,2026-10-07 11:40,26.290001,0.800000,null,0.500000,4.600000
제주남부,126.965858,32.090408,2026-10-07 10:00,351.000000,2.580000,6.500000,1.800000,6.500000
제주남부,126.965858,32.090408,2026-10-07 10:30,343.000000,2.380000,6.500000,1.560000,6.500000
제주남부,126.965858,32.090408,2026-10-07 11:00,349.000000,2.240000,6.500000,1.590000,6.500000
대한해협,129.121246,34.918999,2026-10-07 10:00,52.000000,1.390000,5.000000,0.820000,5.900000
대한해협,129.121246,34.918999,2026-10-07 10:30,49.000000,1.260000,6.000000,0.820000,6.000000
대한해협,129.121246,34.918999,2026-10-07 11:00,50.000000,1.250000,6.500000,0.800000,6.000000
남해동부,128.419022,34.222469,2026-10-07 10:00,0.000000,1.380000,4.500000,0.850000,5.100000
남해동부,128.419022,34.222469,2026-10-07 10:30,342.000000,1.270000,4.500000,0.860000,5.000000


In [152]:
val tableName = "WaveInfo"
val sql = """ INSERT OR REPLACE INTO $tableName
 ( obsvtrNm, lot, lat, obsrvnDt, wvhgt, wvpd, wvdrct, maxWvhgt, maxWvpd )
 VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""".trimIndent()

DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql)?.use { pstmt ->
        df_cleaned.forEach { row ->
            try {
                pstmt.setString(1, row["obsvtrNm"]?.toString())
                pstmt.setString(2, row["lot"]?.toString())
                pstmt.setString(3, row["lat"]?.toString())
                pstmt.setString(4, row["obsrvnDt"]?.toString())
                pstmt.setString(5, row["wvhgt"]?.toString())
                pstmt.setString(6, row["wvpd"]?.toString())
                pstmt.setString(7, row["wvdrct"]?.toString())
                pstmt.setString(8, row["maxWvhgt"]?.toString())
                pstmt.setString(9, row["maxWvpd"]?.toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}